# Event Analysis

Pandas-based analysis patterns for KanKyouKen event data:
- Event type distributions
- Participant activity
- Time series
- Session analysis
- Cohort analysis

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from notebook_setup import setup

client, STUDY_ID, PARTICIPANT_IDS = setup(n_participants=5, n_events=100)

URL:          http://127.0.0.1:54321
Study:        1269cc61-4612-4c6d-926c-c7f7998f4a33
Participants: 5
Events:       100


## Load All Events

In [2]:
import pandas as pd

df = pd.DataFrame([e.to_dict() for e in client.iter_events(study_id=STUDY_ID)])

df["ts"] = pd.to_datetime(df["ts"])
df["date"] = df["ts"].dt.date
df["hour"] = df["ts"].dt.hour
df["day_of_week"] = df["ts"].dt.day_name()
df = df.sort_values("ts").reset_index(drop=True)

print(f"Events: {len(df)}")
print(f"Participants: {df['participant_id'].nunique()}")
print(f"Event types: {df['event_type'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

Events: 100
Participants: 5
Event types: 5
Date range: 2026-01-29 to 2026-02-13


,id,participant_id,study_id,event_type,ts,session_id,app_version,platform,item_id,task_id,created_at,payload_page,payload_duration_ms,date,hour,day_of_week
0,618f1391-b774-4df7-8d69-8860f452ffa9,d0d37f4e-7fdf-4fba-b1c2-32fa4c7ea6d1,1269cc61-4612-4c6d-926c-c7f7998f4a33,form_submit,2026-01-29 19:27:26.154598+00:00,None,None,None,None,None,2026-02-12 09:26:26.992802+00:00,NaN,NaN,2026-01-29,19,Thursday
1,a8d6ada6-ec23-47a0-bee6-83d21a5011fc,d78d4bab-d179-42ee-9403-150a525c317b,1269cc61-4612-4c6d-926c-c7f7998f4a33,form_submit,2026-01-29 20:37:26.154598+00:00,None,None,None,None,None,2026-02-12 09:26:28.097061+00:00,NaN,NaN,2026-01-29,20,Thursday
2,e35c7e9b-e2ac-4ac9-9c6b-00d185172fc8,d0d37f4e-7fdf-4fba-b1c2-32fa4c7ea6d1,1269cc61-4612-4c6d-926c-c7f7998f4a33,login,2026-01-30 05:33:26.154598+00:00,None,None,None,None,None,2026-02-12 09:26:28.722434+00:00,NaN,NaN,2026-01-30,5,Friday
3,2a658a30-e3e4-4c0c-bebd-e76556a0615e,d0d37f4e-7fdf-4fba-b1c2-32fa4c7ea6d1,1269cc61-4612-4c6d-926c-c7f7998f4a33,form_submit,2026-01-30 07:10:26.154598+00:00,None,None,None,None,None,2026-02-12 09:26:26.184877+00:00,NaN,NaN,2026-01-30,7,Friday
4,1811fc3d-08fc-4d03-80ba-496201c67c81,d4b3c963-0627-4707-b594-43d54f08dc02,1269cc61-4612-4c6d-926c-c7f7998f4a33,form_submit,2026-01-30 21:26:26.154598+00:00,None,None,None,None,None,2026-02-12 09:26:28.663586+00:00,NaN,NaN,2026-01-30,21,Friday


## Event Type Distribution

In [3]:
type_counts = df["event_type"].value_counts()
type_pct = (type_counts / len(df) * 100).round(1)

summary = pd.DataFrame({"count": type_counts, "pct": type_pct})
print(summary.to_string())

              count   pct
event_type               
form_submit      27  27.0
logout           22  22.0
button_click     20  20.0
login            16  16.0
page_view        15  15.0


## Participant Activity

In [4]:
participant_stats = (
    df.groupby("participant_id")
    .agg(
        total_events=("id", "count"),
        unique_event_types=("event_type", "nunique"),
        first_event=("ts", "min"),
        last_event=("ts", "max"),
    )
    .reset_index()
)
participant_stats["active_days"] = (
    (participant_stats["last_event"] - participant_stats["first_event"]).dt.total_seconds() / 86400
).round(1)

print("Participant activity summary:")
print(participant_stats[["total_events", "unique_event_types", "active_days"]].describe().round(1))

Participant activity summary:
       total_events  unique_event_types  active_days
count           5.0                 5.0          5.0
mean           20.0                 5.0         13.0
std             2.9                 0.0          1.8
min            17.0                 5.0          9.9
25%            18.0                 5.0         13.3
50%            19.0                 5.0         13.3
75%            22.0                 5.0         14.1
max            24.0                 5.0         14.4


## Time Series

In [5]:
# Daily event counts
daily = df.groupby("date").size().rename("events")
print("Daily events (last 14 days):")
print(daily.tail(14).to_string())
print(f"\nAverage per day: {daily.mean():.1f}")

Daily events (last 14 days):
date
2026-01-31     3
2026-02-01     6
2026-02-02     5
2026-02-03     8
2026-02-04     6
2026-02-05     9
2026-02-06     8
2026-02-07     9
2026-02-08    10
2026-02-09     6
2026-02-10     6
2026-02-11     3
2026-02-12     9
2026-02-13     5

Average per day: 6.2


In [6]:
# Hourly distribution
hourly = df.groupby("hour").size().rename("events")
peak_hour = hourly.idxmax()
print(f"Peak activity hour: {peak_hour}:00 ({hourly[peak_hour]} events)")
print()
print(hourly.to_string())

Peak activity hour: 5:00 (10 events)

hour
0      4
1      8
2      7
3      7
4      7
5     10
6      7
7     10
8      2
17     5
18     5
19     4
20     2
21     9
22     7
23     6


## Session Analysis

Only runs if your events include a  field.

In [7]:
if "session_id" in df.columns and df["session_id"].notna().any():
    sessions = (
        df[df["session_id"].notna()]
        .groupby("session_id")
        .agg(
            participant=("participant_id", "first"),
            events=("id", "count"),
            start=("ts", "min"),
            end=("ts", "max"),
        )
        .reset_index()
    )
    sessions["duration_min"] = (
        (sessions["end"] - sessions["start"]).dt.total_seconds() / 60
    ).round(1)
    print(f"Sessions: {len(sessions)}")
    print(sessions[["events", "duration_min"]].describe().round(1))
else:
    print("No session_id column found in this dataset.")

No session_id column found in this dataset.


## Weekly Cohorts

Group participants by the week of their first event.

In [8]:
first_seen = df.groupby("participant_id")["ts"].min().rename("first_event")
df_cohort = df.join(first_seen, on="participant_id")
df_cohort["cohort_week"] = df_cohort["first_event"].dt.to_period("W")

cohort = (
    df_cohort.groupby("cohort_week")
    .agg(new_participants=("participant_id", "nunique"), total_events=("id", "count"))
    .reset_index()
)
cohort["avg_events"] = (cohort["total_events"] / cohort["new_participants"]).round(1)
print(cohort.to_string(index=False))

          cohort_week  new_participants  total_events  avg_events
2026-01-26/2026-02-01                 4            83        20.8
2026-02-02/2026-02-08                 1            17        17.0


/tmp/ipykernel_67417/3268524513.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_cohort["cohort_week"] = df_cohort["first_event"].dt.to_period("W")


## Payload Field Analysis

If events carry numeric payload fields, summarise them.

In [9]:
payload_cols = [c for c in df.columns if c.startswith("payload_")]
if payload_cols:
    numeric_payload = df[payload_cols].select_dtypes(include="number")
    if not numeric_payload.empty:
        print("Numeric payload fields summary:")
        print(numeric_payload.describe().round(2))
    else:
        print("Payload columns found but none are numeric:", payload_cols)
else:
    print("No payload columns in this dataset.")

Numeric payload fields summary:
       payload_duration_ms
count                15.00
mean               9635.87
std                3442.42
min                1765.00
25%                9143.00
50%                9769.00
75%               10941.00
max               14841.00


## Export

In [10]:
import os
os.makedirs("analysis_output", exist_ok=True)

participant_stats.to_csv("analysis_output/participant_stats.csv", index=False)
daily.to_csv("analysis_output/daily_activity.csv", header=True)
type_counts.to_csv("analysis_output/event_type_counts.csv", header=True)

print("Saved to analysis_output/")

Saved to analysis_output/
